In [1]:
# pip install pandas openpyxl torch transformers sentencepiece uuid
# import nltk
# nltk.download("wordnet")

# Translate Word Pairs Function

For word pairs that have not previously been translated, we will first translate those pairs with a newer model than used in the original SPAML.

In [ ]:
import pandas as pd
from transformers import AutoProcessor, AutoModelForSeq2SeqLM
import random
from pathlib import Path
import json
import uuid
from pathlib import Path
from nltk.corpus import wordnet
import re

# clean words 
def clean_word(text):
    text = text.lower()
    text = re.sub(r"[^\w\s]", "", text)   # remove punctuation
    text = text.strip()
    return text

# translate the english words into the target language
def translate_pairs(df, target_lang, src_lang="eng", batch_size=64):

    """
    Translate cue-target pairs and fix cases where cue == target
    using WordNet synonyms.

    Requires global:
        processor
        model
    """

    # ------------------------------
    # 1 Translate all words
    # ------------------------------

    words = df["en_cue"].tolist() + df["en_target"].tolist()

    translations = []

    for i in range(0, len(words), batch_size):

        batch = words[i:i+batch_size]

        inputs = processor(
            text=batch,
            src_lang=src_lang,
            return_tensors="pt",
            padding=True
        )

        outputs = model.generate(
            **inputs,
            tgt_lang=target_lang,
            max_new_tokens=20
        )

        decoded = processor.batch_decode(
            outputs,
            skip_special_tokens=True
        )

        translations.extend(decoded)

    half = len(df)

    df[f"{target_lang}_cue"] = translations[:half]
    df[f"{target_lang}_target"] = translations[half:]

    # ------------------------------
    # 2 lowercase
    # ------------------------------

    # clean translations
    df[f"{target_lang}_cue"] = df[f"{target_lang}_cue"].apply(clean_word)
    df[f"{target_lang}_target"] = df[f"{target_lang}_target"].apply(clean_word)

    # ------------------------------
    # 3 detect cue == target
    # ------------------------------

    df["needs_review"] = df[f"{target_lang}_cue"] == df[f"{target_lang}_target"]

    df[f"suggested_{target_lang}_cue"] = None

    # ------------------------------
    # 4 generate synonym-based cue suggestions
    # ------------------------------

    for idx in df[df["needs_review"]].index:

        english_cue = df.loc[idx, "en_cue"]
        target_word = df.loc[idx, f"{target_lang}_target"]

        # Get WordNet synonyms
        synsets = wordnet.synsets(english_cue)

        synonyms = set()

        for syn in synsets:
            for lemma in syn.lemmas():
                word = lemma.name().replace("_", " ")
                if word.lower() != english_cue.lower():
                    synonyms.add(word)

        synonyms = list(synonyms)

        suggestion = None

        if synonyms:

            inputs = processor(
                text=synonyms,
                src_lang=src_lang,
                return_tensors="pt",
                padding=True
            )

            outputs = model.generate(
                **inputs,
                tgt_lang=target_lang,
                max_new_tokens=20
            )

            translated_syns = processor.batch_decode(
                outputs,
                skip_special_tokens=True
            )

            translated_syns = [clean_word(w) for w in translated_syns]

            for candidate in translated_syns:
                if candidate != target_word:
                    suggestion = candidate
                    break

        df.loc[idx, f"suggested_{target_lang}_cue"] = suggestion

    # ------------------------------
    # 5 create final cue column
    # ------------------------------

    df[f"final_{target_lang}_cue"] = df[f"{target_lang}_cue"]

    mask = df["needs_review"] & df[f"suggested_{target_lang}_cue"].notna()

    df.loc[mask, f"final_{target_lang}_cue"] = df.loc[
        mask, f"suggested_{target_lang}_cue"
    ]

    # ------------------------------
    # 6 save results
    # ------------------------------

    Path(target_lang).mkdir(exist_ok=True)

    df.to_csv(f"{target_lang}/{target_lang}_word_pairs.csv", index=False)

    print(f"Saved translated pairs to {target_lang}/{target_lang}_word_pairs.csv")
    print(f"{df['needs_review'].sum()} cue==target collisions detected")

    return df

# Libraries and Models

In [ ]:
# Load model once (important for speed)
model_name = "facebook/seamless-m4t-v2-large"

processor = AutoProcessor.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

word_pairs_english = pd.read_csv("en_words.csv")

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [5]:
# first translate the words
# translate_pairs(word_pairs_english, "afr")

translate_pairs(word_pairs_english, "ukr")

Saved translated pairs to ukr/ukr_word_pairs.csv
57 cue==target collisions detected


,en_cue,en_target,ukr_cue,ukr_target,needs_review,suggested_ukr_cue,final_ukr_cue
0,leave,abandon,піти,залишити,False,None,піти
1,spinal,abdominal,спинальний,живота,False,None,спинальний
2,kidnap,abduct,викрасти,викрадення,False,None,викрасти
3,capacity,ability,потужність,здатність,False,None,потужність
4,out,about,вийшов,про,False,None,вийшов
...,...,...,...,...,...,...,...
995,okay,yeah,гаразд,так,False,None,гаразд
996,red,yellow,червоний,жовтий,False,None,червоний
997,today,yesterday,сьогодні,вчора,False,None,сьогодні
998,old,young,старий,молодий,False,None,старий
